# 02 · Entrenamiento de los modelos

Entrena los detectores YOLO del TFM a partir del dataset preparado en el
notebook 01. Puede entrenar un solo modelo o una cola de varios, uno detrás de
otro.

## Guía de lectura y ejecución

| Aspecto | Descripción |
|---|---|
| **Papel en el proyecto** | Entrena los modelos YOLO que se comparan en el TFM. |
| **Cuándo abrirlo** | Para ver la configuración de entrenamiento o volver a entrenar un modelo. |
| **Comportamiento por defecto** | No entrena (`RUN_TRAINING=False`). Para entrenar hace falta una GPU. |
| **Entradas principales** | Dataset preparado por el notebook 01 y `configs/model_registry.yaml`. |
| **Salidas principales** | Una carpeta por entrenamiento en `artifacts/experiments/<id>/` con pesos, métricas y configuración. |
| **Continuación** | `03_DFire_evaluacion_modelos.ipynb`. |

> **Para ejecutarlo:** usar el entorno Docker de notebooks (sección 4 del
> `README.md` principal). Volver a ejecutarlo requiere los resultados de los
> notebooks anteriores; ver `notebooks/README.md`.

## Cómo funciona

- Cada entrenamiento se guarda en su propia carpeta (`artifacts/experiments/<id>/`)
  con los pesos (`best.pt` y `last.pt`), las métricas y la configuración usada.
- Todos los modelos usan la misma configuración base: 100 épocas, batch 16,
  AdamW con tasa de aprendizaje inicial 0,001 y parada temprana tras 20 épocas
  sin mejora.
- Entre un entrenamiento y el siguiente se libera la memoria de la GPU. Si uno
  falla, queda marcado como `failed` y la cola sigue si `CONTINUE_ON_ERROR=True`.
- Este notebook no usa el conjunto de test.

## 1. Parámetros

- `RUN_TRAINING=True`: lanza el entrenamiento (necesita GPU).
- `RESUME_TRAINING=True`: reanuda el entrenamiento interrumpido indicado en
  `EXPERIMENT_ID`.
- `MODEL_KEY` y `TRAINING_PROFILE`: modelo y configuración cuando se entrena uno
  solo. Por defecto, `yolo26s` con `controlled_768`, la configuración del modelo
  final.
- `TRAINING_QUEUE`: lista de entrenamientos para lanzar varios seguidos.

In [ ]:
RUN_TRAINING = False
RESUME_TRAINING = False
MODEL_KEY = "yolo26s"       # modo de un solo modelo cuando TRAINING_QUEUE está vacía
TRAINING_PROFILE = "controlled_768"
TRAINING_QUEUE = [
    # {"model_key": "yolov8s", "training_profile": "controlled_1024"},
    # {"model_key": "yolo26s", "training_profile": "controlled_1024"}
]
CONTINUE_ON_ERROR = True    # intenta el siguiente trabajo y muestra los fallos al final
DATASET_VERSION = "dfire_seed42_val10_v1"
SEED = 42
EXPERIMENT_ID = None         # modo individual; obligatorio para RESUME_TRAINING=True
RESUME_CHECKPOINT = None     # opcional; por defecto usa last.pt del experimento
ALLOW_CPU_TRAINING = False
WORKERS = 4
USE_FAST_LOCAL_DATASET = True
FAST_DATA_ROOT = None       # None usa /workspace/.cache/tfm-datasets en Docker
STAGING_WORKERS = 8


## 2. Entorno, dataset y registro de modelos


In [2]:
from pathlib import Path
import datetime as dt
import gc
import json
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ultralytics import YOLO

PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", "/workspace/TFM"))
if not PROJECT_ROOT.is_dir():
    PROJECT_ROOT = Path("C:/Users/elitr/Documents/UPM Data/TFM")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import tfm_pipeline as pipeline

PROJECT_ROOT = pipeline.project_root()
contract = pipeline.validate_prepared_dataset(PROJECT_ROOT, DATASET_VERSION)
registry = pipeline.load_model_registry(PROJECT_ROOT)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
environment = pipeline.environment_snapshot()

RESERVED_OVERRIDES = {"data", "device", "workers", "project", "name", "exist_ok", "resume", "seed"}

def normalise_job(raw_job, position):
    if not isinstance(raw_job, dict):
        raise TypeError(f"El trabajo {position} debe ser un diccionario.")
    model_key = raw_job.get("model_key", MODEL_KEY)
    profile = raw_job.get("training_profile", TRAINING_PROFILE)
    job_seed = int(raw_job.get("seed", SEED))
    overrides = dict(raw_job.get("overrides", {}))
    forbidden = RESERVED_OVERRIDES.intersection(overrides)
    if forbidden:
        raise ValueError(f"Overrides reservados en el trabajo {position}: {sorted(forbidden)}")
    if model_key not in registry["models"]:
        raise KeyError(f"Modelo desconocido en el trabajo {position}: {model_key!r}")
    if profile not in registry["profiles"]:
        raise KeyError(f"Perfil desconocido en el trabajo {position}: {profile!r}")

    model_config = dict(registry["models"][model_key])
    train_config = dict(registry["profiles"][profile])
    train_config.update(model_config.get("overrides", {}))
    train_config.update(overrides)
    return {
        "position": position,
        "model_key": model_key,
        "training_profile": profile,
        "seed": job_seed,
        "experiment_id": raw_job.get("experiment_id"),
        "model_config": model_config,
        "train_config": train_config,
        "weights": pipeline.resolve_model_weights(model_config, PROJECT_ROOT),
    }

queue_source = TRAINING_QUEUE or [{
    "model_key": MODEL_KEY,
    "training_profile": TRAINING_PROFILE,
    "seed": SEED,
    "experiment_id": EXPERIMENT_ID,
}]
planned_jobs = [normalise_job(job, index) for index, job in enumerate(queue_source, start=1)]

plan_rows = []
for job in planned_jobs:
    plan_rows.append({
        "orden": job["position"],
        "modelo": job["model_key"],
        "perfil": job["training_profile"],
        "imgsz": job["train_config"].get("imgsz"),
        "epochs": job["train_config"].get("epochs"),
        "batch": job["train_config"].get("batch"),
        "seed": job["seed"],
        "experiment_id": job["experiment_id"] or "automático",
    })

display(pd.DataFrame(plan_rows))
display(pd.Series({
    "dataset_version": DATASET_VERSION,
    "dataset_manifest_sha256": pipeline.sha256_file(contract["manifest_path"]),
    "device": environment["gpu_name"] or "CPU",
    "trabajos_en_cola": len(planned_jobs),
    "continuar_si_falla": CONTINUE_ON_ERROR,
}, name="valor").to_frame())


,orden,modelo,perfil,imgsz,epochs,batch,seed,experiment_id
0,1,yolov8s,controlled_768,768,100,16,42,automático


,valor
dataset_version,dfire_seed42_val10_v1
dataset_manifest_sha256,97442af38322abe1bf5178bd6b92c30971d9f354eca319...
device,NVIDIA GeForce RTX 5070
trabajos_en_cola,1
continuar_si_falla,True


## 3. Comprobaciones previas


In [3]:
if (RUN_TRAINING or RESUME_TRAINING) and not torch.cuda.is_available() and not ALLOW_CPU_TRAINING:
    raise RuntimeError(
        "Entrenamiento bloqueado: CUDA no está disponible. La preparación y la "
        "inspección sí pueden ejecutarse en CPU."
    )
if RUN_TRAINING and RESUME_TRAINING:
    raise ValueError("Activa RUN_TRAINING o RESUME_TRAINING, pero no ambos.")
if RESUME_TRAINING and not EXPERIMENT_ID:
    raise ValueError("Para reanudar debes fijar EXPERIMENT_ID.")
if RESUME_TRAINING and TRAINING_QUEUE:
    raise ValueError("La reanudación admite un experimento individual; vacía TRAINING_QUEUE.")
if not planned_jobs:
    raise ValueError("No hay trabajos de entrenamiento configurados.")

available = pipeline.list_experiments(PROJECT_ROOT)
if not available.empty:
    display(available[[
        column for column in (
            "experiment_id", "model_key", "status", "created_utc", "best_model_exists"
        ) if column in available.columns
    ]])
print("Preflight correcto. El test no se ha cargado.")


,experiment_id,model_key,status,created_utc,best_model_exists
0,legacy_yolov8s_baseline,yolov8s,complete,2026-08-29T21:31:47+00:00,True
1,yolo26n_dfire_seed42_20260831T074308Z,yolo26n,complete,2026-08-31T07:43:08.356954+00:00,True
2,yolo26s_dfire_seed42_20260830T225658Z,yolo26s,complete,2026-08-30T22:56:58.980721+00:00,True
3,yolov8n_dfire_seed42_20260830T174159Z,yolov8n,complete,2026-08-30T17:41:59.929107+00:00,True
4,yolov8s_dfire_seed42_20260911T210842Z,yolov8s,failed,2026-09-11T21:08:42.130763+00:00,True
5,yolov8s_dfire_seed42_20260912T000125Z,yolov8s,complete,2026-09-12T00:01:25.602683+00:00,True


Preflight correcto. El test no se ha cargado.


## 4. Entrenar o reanudar

In [4]:
completed_runs = []
failed_runs = []

def set_random_seed(job_seed):
    random.seed(job_seed)
    np.random.seed(job_seed)
    torch.manual_seed(job_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(job_seed)

def next_experiment_id(job):
    requested = job["experiment_id"]
    base_id = requested or pipeline.utc_experiment_id(job["model_key"], job["seed"])
    candidate = base_id
    suffix = 2
    while (pipeline.experiments_root(PROJECT_ROOT) / candidate).exists():
        if requested:
            raise FileExistsError(
                f"El experimento ya existe: {candidate}. Cambia experiment_id o reanúdalo."
            )
        candidate = f"{base_id}_{suffix:02d}"
        suffix += 1
    return candidate

if RESUME_TRAINING:
    experiment = pipeline.resolve_experiment(
        experiment_id=EXPERIMENT_ID, root=PROJECT_ROOT
    )
    checkpoint = Path(RESUME_CHECKPOINT or experiment["last_model"])
    if not checkpoint.exists():
        raise FileNotFoundError(f"No existe el checkpoint: {checkpoint}")
    print(f"Reanudando {EXPERIMENT_ID} desde {checkpoint}")
    training_model = YOLO(str(checkpoint))
    training_result = training_model.train(resume=True)
    run_dir = Path(training_result.save_dir)
    descriptor_path = Path(experiment["experiment_root"]) / "experiment.json"
    saved = pipeline.read_json(descriptor_path)
    saved.update({
        "status": "complete",
        "completed_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "training_run_dir_rel": pipeline.project_relative(run_dir, PROJECT_ROOT),
        "best_model_rel": pipeline.project_relative(run_dir / "weights" / "best.pt", PROJECT_ROOT),
        "last_model_rel": pipeline.project_relative(run_dir / "weights" / "last.pt", PROJECT_ROOT),
    })
    pipeline.write_json_atomic(descriptor_path, saved)
    completed_runs.append(pipeline.resolve_descriptor_paths(saved, PROJECT_ROOT))

elif RUN_TRAINING:
    staging_base = (
        Path(FAST_DATA_ROOT) if FAST_DATA_ROOT
        else pipeline.default_fast_data_root(PROJECT_ROOT)
        if USE_FAST_LOCAL_DATASET
        else PROJECT_ROOT / "artifacts" / "runtime_datasets"
    )
    staged_dataset = pipeline.stage_prepared_dataset(
        contract, fast_base=staging_base, workers=STAGING_WORKERS
    )

    for job in planned_jobs:
        experiment_id = next_experiment_id(job)
        experiment_root = pipeline.experiments_root(PROJECT_ROOT) / experiment_id
        experiment_root.mkdir(parents=True)
        runtime_yaml = pipeline.write_runtime_data_yaml(
            contract, experiment_root / "data_runtime.yaml", staged_dataset
        )
        expected_run_dir = experiment_root / "train"
        model_config = job["model_config"]
        descriptor_path = experiment_root / "experiment.json"
        experiment = {
            "schema_version": 1,
            "experiment_id": experiment_id,
            "queue_position": job["position"],
            "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "status": "running",
            "model_key": job["model_key"],
            "model_family": model_config["family"],
            "model_scale": model_config["scale"],
            "initial_weights": job["weights"],
            "dataset_version": DATASET_VERSION,
            "dataset_manifest_sha256": pipeline.sha256_file(contract["manifest_path"]),
            "profile": job["training_profile"],
            "seed": job["seed"],
            "train_config": job["train_config"],
            "training_run_dir_rel": pipeline.project_relative(expected_run_dir, PROJECT_ROOT),
            "best_model_rel": pipeline.project_relative(expected_run_dir / "weights" / "best.pt", PROJECT_ROOT),
            "last_model_rel": pipeline.project_relative(expected_run_dir / "weights" / "last.pt", PROJECT_ROOT),
        }
        pipeline.write_json_atomic(descriptor_path, experiment)
        pipeline.write_json_atomic(experiment_root / "environment.json", environment)

        training_model = None
        try:
            print(f"[{job['position']}/{len(planned_jobs)}] Iniciando {experiment_id}")
            set_random_seed(job["seed"])
            training_model = YOLO(job["weights"])
            training_result = training_model.train(
                data=str(runtime_yaml),
                device=DEVICE,
                workers=WORKERS,
                project=str(experiment_root),
                name="train",
                exist_ok=False,
                seed=job["seed"],
                **job["train_config"],
            )
            run_dir = Path(training_result.save_dir)
            saved = pipeline.read_json(descriptor_path)
            saved.update({
                "status": "complete",
                "completed_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
                "training_run_dir_rel": pipeline.project_relative(run_dir, PROJECT_ROOT),
                "best_model_rel": pipeline.project_relative(run_dir / "weights" / "best.pt", PROJECT_ROOT),
                "last_model_rel": pipeline.project_relative(run_dir / "weights" / "last.pt", PROJECT_ROOT),
            })
            pipeline.write_json_atomic(descriptor_path, saved)
            completed_runs.append(pipeline.resolve_descriptor_paths(saved, PROJECT_ROOT))
            print(f"Completado: {experiment_id}")
        except BaseException as exc:
            saved = pipeline.read_json(descriptor_path)
            saved.update({
                "status": "failed",
                "failed_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            })
            pipeline.write_json_atomic(descriptor_path, saved)
            failed_runs.append({
                "experiment_id": experiment_id,
                "model_key": job["model_key"],
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            })
            print(f"Falló {experiment_id}: {type(exc).__name__}: {exc}")
            if not CONTINUE_ON_ERROR or isinstance(exc, (KeyboardInterrupt, SystemExit)):
                raise
        finally:
            del training_model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
else:
    print("Modo seguro: no se ha iniciado ningún entrenamiento.")


[1/1] Iniciando yolov8s_dfire_seed42_20260912T104424Z
New https://pypi.org/project/ultralytics/8.4.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.132 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/TFM/artifacts/experiments/yolov8s_dfire_seed42_20260912T104424Z/data_runtime.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0

## 5. Resumen de los entrenamientos

In [5]:
if completed_runs:
    display(pd.DataFrame([{
        "experiment_id": run["experiment_id"],
        "modelo": run["model_key"],
        "estado": run["status"],
        "best.pt": run["best_model"],
    } for run in completed_runs]))
if failed_runs:
    display(pd.DataFrame(failed_runs))
if not completed_runs and not failed_runs:
    print("Sin cambios: configura RUN_TRAINING=True cuando quieras lanzar el modelo seleccionado.")
else:
    print(f"Cola finalizada: {len(completed_runs)} completos y {len(failed_runs)} fallidos.")
    print("La evaluación en validación corresponde al notebook 03; el test permanece reservado.")


,experiment_id,modelo,estado,best.pt
0,yolov8s_dfire_seed42_20260912T104424Z,yolov8s,complete,/workspace/TFM/artifacts/experiments/yolov8s_d...


Cola finalizada: 1 completos y 0 fallidos.
La evaluación en validación corresponde al notebook 03; el test permanece reservado.


## 6. Añadir otro modelo

Para probar otro modelo compatible con Ultralytics, basta con añadir una
entrada en `configs/model_registry.yaml` y seleccionarla en `MODEL_KEY` o en
`TRAINING_QUEUE`; no hace falta modificar este notebook.

Cada entrada de `TRAINING_QUEUE` admite `model_key`, `training_profile`, `seed`,
`experiment_id` y `overrides` (por ejemplo, `overrides={"imgsz": 1024}` para una
prueba puntual). Para comparar modelos entre sí conviene usar siempre el mismo
perfil de entrenamiento.